# Streamlit & Gradio: ML Demo Apps

## Streamlit
Streamlit turns Python scripts into interactive web apps with no HTML/CSS/JS.

## Gradio
Gradio creates ML demos with auto-generated UIs, tightly integrated with Hugging Face.

## When to Use Each
| | Streamlit | Gradio |
|--|-----------|--------|
| **Best for** | Full dashboards, data apps | Quick ML demos, HF Spaces |
| **UI control** | High | Medium |
| **HF integration** | Moderate | Native |
| **Learning curve** | Low | Very low |

In [1]:
# Streamlit complete app (save as app.py and run: streamlit run app.py)
STREAMLIT_APP = '''
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib
import io

# Page configuration
st.set_page_config(
    page_title="ML Dashboard",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Cache model loading (persists across reruns)
@st.cache_resource
def load_model():
    data = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    return model, data.feature_names, data.target_names

# Cache data processing (recomputes when inputs change)
@st.cache_data
def load_data():
    data = load_iris()
    return pd.DataFrame(data.data, columns=data.feature_names)

# Sidebar
st.sidebar.title("⚙️ Configuration")
dataset = st.sidebar.selectbox("Dataset", ["Iris", "Breast Cancer"])
n_estimators = st.sidebar.slider("Number of Trees", 10, 500, 100)
show_raw = st.sidebar.checkbox("Show Raw Data", False)

# Main page
st.title("🤖 ML Classification Dashboard")
st.markdown("Interactive ML model training and evaluation")

# Layout with columns
col1, col2, col3 = st.columns(3)
with col1:
    st.metric("Model", "Random Forest")
with col2:
    st.metric("Trees", n_estimators)
with col3:
    st.metric("Dataset", dataset)

# Tabs
tab1, tab2, tab3 = st.tabs(["📊 Data", "🏋️ Training", "🎯 Prediction"])

with tab1:
    df = load_data()
    st.subheader("Dataset Overview")
    col1, col2 = st.columns(2)
    with col1:
        st.write("**Shape:**", df.shape)
        st.write("**Missing values:**", df.isnull().sum().sum())
    with col2:
        st.write("**Statistics:**")
        st.dataframe(df.describe(), use_container_width=True)
    
    if show_raw:
        st.dataframe(df, use_container_width=True)
    
    # Correlation heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(df.corr(), annot=True, cmap="coolwarm", ax=ax)
    st.pyplot(fig)
    plt.close()

with tab2:
    if st.button("🚀 Train Model", type="primary"):
        with st.spinner("Training..."):
            data = load_iris()
            X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
            model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            acc = accuracy_score(y_te, y_pred)
            cm = confusion_matrix(y_te, y_pred)
            
            st.success(f"✅ Model trained! Accuracy: {acc:.4f}")
            st.progress(acc)
            
            fig, ax = plt.subplots(figsize=(6, 4))
            sns.heatmap(cm, annot=True, fmt=\'d\', cmap=\'Blues\',
                       xticklabels=data.target_names, yticklabels=data.target_names, ax=ax)
            ax.set_xlabel("Predicted"); ax.set_ylabel("True")
            st.pyplot(fig); plt.close()
            
            # Feature importance
            fi_df = pd.DataFrame({\'feature\': data.feature_names, \'importance\': model.feature_importances_})
            fi_df = fi_df.sort_values(\'importance\', ascending=False)
            st.bar_chart(fi_df.set_index(\'feature\'))
            
            st.session_state[\'model\'] = model
            st.session_state[\'feature_names\'] = data.feature_names

with tab3:
    st.subheader("Make Predictions")
    if \'model\' in st.session_state:
        model = st.session_state[\'model\']
        feature_names = st.session_state[\'feature_names\']
        
        inputs = []
        cols = st.columns(len(feature_names))
        for i, (col, feat) in enumerate(zip(cols, feature_names)):
            val = col.number_input(feat, value=5.0, step=0.1, key=f\'feat_{i}\')
            inputs.append(val)
        
        if st.button("🎯 Predict"):
            prediction = model.predict([inputs])[0]
            proba = model.predict_proba([inputs])[0]
            iris = load_iris()
            class_name = iris.target_names[prediction]
            
            st.success(f"**Predicted class: {class_name}**")
            prob_df = pd.DataFrame({\'Class\': iris.target_names, \'Probability\': proba})
            st.bar_chart(prob_df.set_index(\'Class\'))
        
        # File upload for batch prediction
        uploaded = st.file_uploader("Upload CSV for batch prediction", type=[\'csv\'])
        if uploaded:
            batch_df = pd.read_csv(uploaded)
            preds = model.predict(batch_df.values)
            batch_df[\'prediction\'] = preds
            st.dataframe(batch_df)
            csv = batch_df.to_csv(index=False)
            st.download_button("Download Predictions", csv, "predictions.csv", "text/csv")
    else:
        st.info("Please train a model first in the Training tab")
'''

with open('/tmp/streamlit_app.py', 'w') as f:
    f.write(STREAMLIT_APP)
print('Streamlit app saved to /tmp/streamlit_app.py')
print('Run with: streamlit run /tmp/streamlit_app.py')

Streamlit app saved to /tmp/streamlit_app.py
Run with: streamlit run /tmp/streamlit_app.py


In [2]:
# Gradio complete app
GRADIO_APP = '''
import gradio as gr
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import pipeline as hf_pipeline
import matplotlib.pyplot as plt
import io
from PIL import Image

# --- Model 1: Iris Classifier ---
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

def classify_iris(sepal_length, sepal_width, petal_length, petal_width):
    features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    prediction = clf.predict(features)[0]
    probabilities = clf.predict_proba(features)[0]
    class_name = iris.target_names[prediction]
    return {cls: float(prob) for cls, prob in zip(iris.target_names, probabilities)}

# --- Model 2: Text Sentiment ---
# sentiment_pipe = hf_pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
def analyze_sentiment(text):
    # Demo without loading model
    import random
    labels = ["POSITIVE", "NEGATIVE"]
    label = random.choice(labels)
    score = random.uniform(0.7, 0.99)
    return {"POSITIVE": score if label == "POSITIVE" else 1-score,
            "NEGATIVE": 1-score if label == "POSITIVE" else score}

# --- App with Blocks API ---
with gr.Blocks(title="ML Demo", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 ML Demo Application")
    gr.Markdown("Interactive demos for different ML models")
    
    with gr.Tab("🌸 Iris Classifier"):
        gr.Markdown("Classify iris flowers based on measurements")
        with gr.Row():
            with gr.Column():
                sepal_l = gr.Slider(4.0, 8.0, value=5.1, label="Sepal Length (cm)")
                sepal_w = gr.Slider(2.0, 4.5, value=3.5, label="Sepal Width (cm)")
                petal_l = gr.Slider(1.0, 7.0, value=1.4, label="Petal Length (cm)")
                petal_w = gr.Slider(0.1, 2.5, value=0.2, label="Petal Width (cm)")
                iris_btn = gr.Button("Classify", variant="primary")
            with gr.Column():
                iris_output = gr.Label(label="Prediction", num_top_classes=3)
        iris_btn.click(classify_iris, inputs=[sepal_l, sepal_w, petal_l, petal_w], outputs=iris_output)
    
    with gr.Tab("💬 Sentiment Analysis"):
        gr.Markdown("Analyze sentiment of text")
        text_input = gr.Textbox(placeholder="Enter text here...", lines=3, label="Input Text")
        sent_output = gr.Label(label="Sentiment", num_top_classes=2)
        gr.Button("Analyze", variant="primary").click(analyze_sentiment, inputs=text_input, outputs=sent_output)
        gr.Examples(["I love this product!", "This is terrible.", "It was okay I guess"], inputs=text_input)
    
    with gr.Tab("📁 Batch Prediction"):
        gr.Markdown("Upload CSV for batch iris classification")
        file_input = gr.File(label="Upload CSV", file_types=[".csv"])
        file_output = gr.File(label="Download Results")
        
        def batch_predict(file):
            df = pd.read_csv(file.name)
            preds = clf.predict(df.values[:, :4])
            df[\'prediction\'] = [iris.target_names[p] for p in preds]
            out_path = "/tmp/predictions.csv"
            df.to_csv(out_path, index=False)
            return out_path
        
        gr.Button("Run Batch Prediction").click(batch_predict, inputs=file_input, outputs=file_output)

demo.launch(share=False)  # share=True creates public URL via HF tunnel
'''

with open('/tmp/gradio_app.py', 'w') as f:
    f.write(GRADIO_APP)
print('Gradio app saved to /tmp/gradio_app.py')
print('Run with: python /tmp/gradio_app.py')

Gradio app saved to /tmp/gradio_app.py
Run with: python /tmp/gradio_app.py


## Deploying on Hugging Face Spaces

```bash
# Create a new Space
# 1. Go to huggingface.co/new-space
# 2. Choose SDK: Gradio or Streamlit
# 3. Clone repo
git clone https://huggingface.co/spaces/username/my-ml-demo
cd my-ml-demo

# Add your app.py + requirements.txt
cat > requirements.txt << 'EOF'
gradio>=4.0.0
scikit-learn
pandas
numpy
transformers
torch
EOF

# Push
git add . && git commit -m 'Add demo' && git push
# Space automatically builds and deploys!
```

## Streamlit Cloud Deployment

```bash
# 1. Push code to GitHub
# 2. Go to share.streamlit.io
# 3. Connect GitHub repo
# 4. Set main file path (app.py)
# Automatic deployment on push!

# requirements.txt
streamlit>=1.28.0
scikit-learn
pandas
matplotlib
seaborn
```

## Additional Learning Resources

### Streamlit
- [Streamlit Docs](https://docs.streamlit.io/)
- [Streamlit Gallery](https://streamlit.io/gallery) Inspiration
- [Streamlit Components](https://streamlit.io/components) Community extensions

### Gradio
- [Gradio Docs](https://www.gradio.app/docs/)
- [Gradio Guides](https://www.gradio.app/guides)
- [Hugging Face Spaces](https://huggingface.co/spaces) Host for free

### Deployment
- [HF Spaces Docs](https://huggingface.co/docs/hub/spaces)
- [Streamlit Cloud](https://streamlit.io/cloud)